# 02 — Model Training & Baseline Comparison

Train ARIMA, Prophet, and XGBoost baselines locally, then train TFT on Kaggle GPU.

**Baseline results (local, already computed):**

| Model | Mean MAE | Median MAE | Mean MAPE | Median MAPE |
|-------|---------|------------|-----------|-------------|
| ARIMA | 42.81 | 20.62 | 5.03% | 3.52% |
| XGBoost | 53.35 | 25.34 | 5.63% | 4.35% |
| Prophet | 60.65 | 30.63 | 6.87% | 5.36% |

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

## 1. Run baselines (if not already computed)

In [ ]:
df = pd.read_csv('../data/modelling_dataset.csv')
print(f'Modelling dataset: {df.shape}')
print(f'Split: {df["split"].value_counts().to_dict()}')

In [ ]:
# ARIMA
from src.models.arima_baseline import run_arima_baseline
arima_results = run_arima_baseline(df)
arima_results.to_csv('../outputs/arima_predictions.csv', index=False)
print(f'ARIMA: {arima_results.shape[0]} predictions')

In [ ]:
# XGBoost
from src.models.xgboost_ts import run_xgboost_baseline
xgb_results = run_xgboost_baseline(df)
xgb_results.to_csv('../outputs/xgboost_predictions.csv', index=False)
print(f'XGBoost: {xgb_results.shape[0]} predictions')

In [ ]:
# Prophet
from src.models.prophet_model import run_prophet_baseline
prophet_results = run_prophet_baseline(df, use_regressors=True)
prophet_results.to_csv('../outputs/prophet_predictions.csv', index=False)
print(f'Prophet: {prophet_results.shape[0]} predictions')

## 2. Compare baselines

In [ ]:
from src.analyse.comparison import (
    compute_metrics, summary_table, metrics_by_cause, metrics_by_country,
    plot_model_comparison_bar, plot_metric_by_cause, plot_forecast_vs_actual
)

arima = pd.read_csv('../outputs/arima_predictions.csv')
xgb = pd.read_csv('../outputs/xgboost_predictions.csv')
prophet = pd.read_csv('../outputs/prophet_predictions.csv')

all_results = pd.concat([arima, xgb, prophet], ignore_index=True)
metrics = compute_metrics(all_results)

print('=== OVERALL ===')
display(summary_table(metrics))
print()
print('=== BY COUNTRY ===')
display(metrics_by_country(metrics))

In [ ]:
plot_model_comparison_bar(metrics, metric='mae', save_path='../outputs/figures/model_comparison_mae.png')
plt.show()

In [ ]:
plot_metric_by_cause(metrics, metric='mae', save_path='../outputs/figures/model_comparison_by_cause.png')
plt.show()

In [ ]:
plot_forecast_vs_actual(all_results, df, save_path='../outputs/figures/forecast_vs_actual.png')
plt.show()

## 3. TFT Training (Kaggle GPU)

The TFT requires GPU training. Copy the cells below into a Kaggle notebook with GPU enabled.

Upload `data/modelling_dataset.csv` as a Kaggle dataset first.

In [ ]:
# === KAGGLE GPU NOTEBOOK CODE ===
# !pip install pytorch-forecasting pytorch-lightning
#
# import pandas as pd
# from src.models.tft import train_tft, build_timeseries_dataset
#
# df = pd.read_csv('/kaggle/input/gbd-forecasting/modelling_dataset.csv')
#
# trained = train_tft(
#     df,
#     hparams={
#         'hidden_size': 64,
#         'attention_head_size': 4,
#         'dropout': 0.1,
#         'learning_rate': 1e-3,
#         'batch_size': 64,
#         'max_epochs': 100,
#         'patience': 10,
#     },
#     max_prediction_length=5,
# )